# (EX) News article processing (with ML pipeline)

# `agnews` Dataset

In [2]:
!curl https://raw.githubusercontent.com/mosesyhc/de300-2025sp-class/refs/heads/main/agnews.csv -O

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 29.3M  100 29.3M    0     0  9148k      0  0:00:03  0:00:03 --:--:-- 9147k


# Pipelining with PySpark MLlib

In [3]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline # pipeline to transform data


In [4]:
spark = (SparkSession.builder
         .master("local[*]")
         .appName("AG news")
         .getOrCreate()
        )
sc = spark.sparkContext

The operation couldn’t be completed. Unable to locate a Java Runtime.
Please visit http://www.java.com for information on installing Java.

/opt/anaconda3/lib/python3.12/site-packages/pyspark/bin/spark-class: line 97: CMD: bad array subscript
head: illegal line count -- -1


PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [30]:
# load dataset
df = spark.read.csv("agnews.csv", inferSchema=True, header=True)
df.show(5)

+-----------+--------------------+--------------------+
|Class Index|               Title|         Description|
+-----------+--------------------+--------------------+
|          3|Wall St. Bears Cl...|Reuters - Short-s...|
|          3|Carlyle Looks Tow...|Reuters - Private...|
|          3|Oil and Economy C...|Reuters - Soaring...|
|          3|Iraq Halts Oil Ex...|Reuters - Authori...|
|          3|Oil prices soar t...|AFP - Tearaway wo...|
+-----------+--------------------+--------------------+
only showing top 5 rows


# Arrange columns

In [31]:
from pyspark.sql.functions import concat_ws, col # to concatinate cols

# renaming 'Class Index' col to 'label'
df = df.withColumnRenamed('Class Index', 'label') # add label
# Assign the result of withColumn back to df to update it with the new 'text' column
df = df.withColumn('text', concat_ws(' ', col('Title'), col('Description'))) # add a new column called text

# concatenating texts
df = df.select('label', 'text')

df.show(10)

+-----+--------------------+
|label|                text|
+-----+--------------------+
|    3|Wall St. Bears Cl...|
|    3|Carlyle Looks Tow...|
|    3|Oil and Economy C...|
|    3|Iraq Halts Oil Ex...|
|    3|Oil prices soar t...|
|    3|Stocks End Up, Bu...|
|    3|Money Funds Fell ...|
|    3|Fed minutes show ...|
|    3|Safety Net (Forbe...|
|    3|Wall St. Bears Cl...|
+-----+--------------------+
only showing top 10 rows


# Tokenize

In [32]:
from pyspark.ml.feature import RegexTokenizer # tokenizer

# convert sentences to list of words
tokenizer = RegexTokenizer(inputCol="text", outputCol="words", pattern="\\W") # want to tokenize text column and make it into words

# apply tokenizer to dataframe
df = tokenizer.transform(df)

df.show(5)

+-----+--------------------+--------------------+
|label|                text|               words|
+-----+--------------------+--------------------+
|    3|Wall St. Bears Cl...|[wall, st, bears,...|
|    3|Carlyle Looks Tow...|[carlyle, looks, ...|
|    3|Oil and Economy C...|[oil, and, econom...|
|    3|Iraq Halts Oil Ex...|[iraq, halts, oil...|
|    3|Oil prices soar t...|[oil, prices, soa...|
+-----+--------------------+--------------------+
only showing top 5 rows


# Stopwords

In [33]:
from pyspark.ml.feature import StopWordsRemover

stopwords_remover = StopWordsRemover(inputCol="words", outputCol="filtered")

# remove stopwords
df = stopwords_remover.transform(df)

df.select(['label', 'words', 'filtered']).show(5)

+-----+--------------------+--------------------+
|label|               words|            filtered|
+-----+--------------------+--------------------+
|    3|[wall, st, bears,...|[wall, st, bears,...|
|    3|[carlyle, looks, ...|[carlyle, looks, ...|
|    3|[oil, and, econom...|[oil, economy, cl...|
|    3|[iraq, halts, oil...|[iraq, halts, oil...|
|    3|[oil, prices, soa...|[oil, prices, soa...|
+-----+--------------------+--------------------+
only showing top 5 rows


# Term frequency, Inverse document frequency

In [37]:
from pyspark.ml.feature import HashingTF

# calculate term frequency in each article (row)
hashing_tf = HashingTF(inputCol='filtered', outputCol='raw_features', numFeatures=16384) # 16384 is a default

featurized_data = hashing_tf.transform(df)

featurized_data.show(5)


+-----+--------------------+--------------------+--------------------+--------------------+
|label|                text|               words|            filtered|        raw_features|
+-----+--------------------+--------------------+--------------------+--------------------+
|    3|Wall St. Bears Cl...|[wall, st, bears,...|[wall, st, bears,...|(16384,[906,1198,...|
|    3|Carlyle Looks Tow...|[carlyle, looks, ...|[carlyle, looks, ...|(16384,[98,156,14...|
|    3|Oil and Economy C...|[oil, and, econom...|[oil, economy, cl...|(16384,[338,1612,...|
|    3|Iraq Halts Oil Ex...|[iraq, halts, oil...|[iraq, halts, oil...|(16384,[180,2731,...|
|    3|Oil prices soar t...|[oil, prices, soa...|[oil, prices, soa...|(16384,[1546,1752...|
+-----+--------------------+--------------------+--------------------+--------------------+
only showing top 5 rows


In [39]:
from pyspark.ml.feature import IDF

idf = IDF(inputCol='raw_features', outputCol='features')


# inverse document frequency
idf_vectorizer = idf.fit(featurized_data) # have to fit to the data
rescaled_data = idf_vectorizer.transform(featurized_data) # using the model on the data

# can also do this
#idf.fit(featurized_data).transform(featurized_data).show(5)

rescaled_data.show(10)

+-----+--------------------+--------------------+--------------------+--------------------+--------------------+
|label|                text|               words|            filtered|        raw_features|            features|
+-----+--------------------+--------------------+--------------------+--------------------+--------------------+
|    3|Wall St. Bears Cl...|[wall, st, bears,...|[wall, st, bears,...|(16384,[906,1198,...|(16384,[906,1198,...|
|    3|Carlyle Looks Tow...|[carlyle, looks, ...|[carlyle, looks, ...|(16384,[98,156,14...|(16384,[98,156,14...|
|    3|Oil and Economy C...|[oil, and, econom...|[oil, economy, cl...|(16384,[338,1612,...|(16384,[338,1612,...|
|    3|Iraq Halts Oil Ex...|[iraq, halts, oil...|[iraq, halts, oil...|(16384,[180,2731,...|(16384,[180,2731,...|
|    3|Oil prices soar t...|[oil, prices, soa...|[oil, prices, soa...|(16384,[1546,1752...|(16384,[1546,1752...|
|    3|Stocks End Up, Bu...|[stocks, end, up,...|[stocks, end, nea...|(16384,[9,555,114...|(1638

In [40]:
# 16384 is all possible words we keep track of.
# raw features: keeps count of raw features
# features: factors in how many times the word shows up. if a rarer word, then it is probably more important

rescaled_data.select('raw_features').show(2, truncate=False)
rescaled_data.select('features').show(2, truncate=False)

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|raw_features                                                                                                                                                                                                                                  |
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|(16384,[906,1198,4756,5540,5638,5831,6235,7372,8905,11170,11790,12343,12766,13441,14118,16126],[1.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0])                                                                             |
|(16384,[98,156,1445,1913,2309,2586,

# Training a multinomial logistic regression

In [41]:
# split data into training and testing
(train, test) = rescaled_data.randomSplit([0.75, 0.25], seed=42)

In [42]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol = 'features',
                        labelCol = 'label',
                        family = 'multinomial',
                        regParam=0.3,
                        elasticNetParam=0,
                        maxIter=20)

lrModel = lr.fit(train)

# Prediction and evaluation

In [43]:
# predict on test data
predictions = lrModel.transform(test)

In [ ]:
from pyspark.sql.functions import avg
from pyspark.sql.types import FloatType

# accuracy calculation

In [ ]:
from pyspark.mllib.evaluation import MulticlassMetrics
# labels = ["World", "Sports", "Business","Science"]

# take only the predictions
preds_and_labels =


In [ ]:
# confusion matrix
metrics =

# Pipelining, from start to finish

In [ ]:
# load dataset
df = spark.read.csv("agnews.csv", inferSchema=True, header=True)

def arrangeColumns(df):
  # Renaming 'Class Index' col to 'label'
  df = df.withColumnRenamed('Class Index', 'label')

  # Add a new column 'text' by joining 'Title' and 'Description'
  df = df.withColumn("text", concat_ws(" ", "Title", 'Description'))

  # Select new text feature and labels
  df = df.select('label', 'text')
  return df

df = arrangeColumns(df)

# tokenizer
tokenizer = RegexTokenizer(inputCol="text", outputCol="words", pattern="\\W")

# stopwords
stopwords_remover = StopWordsRemover(inputCol="words", outputCol="filtered")

# term frequency
hashing_tf = HashingTF(inputCol="filtered",
                       outputCol="raw_features",
                       numFeatures=16384)

# Inverse Document Frequency
idf = IDF(inputCol="raw_features", outputCol="features")

# model
lr = LogisticRegression(featuresCol='features',
                        labelCol='label',
                        family="multinomial",
                        regParam=0.3,
                        elasticNetParam=0,
                        maxIter=20)



In [ ]:
# Put everything in pipeline
pipeline = Pipeline(stages=[tokenizer,
                            stopwords_remover,
                            hashing_tf,
                            idf,
                            lr])

# Fit the pipeline to training documents.
pipelineFit = pipeline.fit(df)

# transform and train
dataset = pipelineFit.transform(df)